In [1]:
import csv
import sqlite3

from bibliometric_rag_assistant.config import DB_PATH, PROJECT_ROOT

output_path = (
    PROJECT_ROOT
    / "data"
    / "curated"
    / "pubmed_sample_screening.csv"
)

output_path.parent.mkdir(parents=True, exist_ok=True)

connection = sqlite3.connect(DB_PATH)
connection.row_factory = sqlite3.Row

records = connection.execute(
    """
    SELECT
        run_id,
        pmid,
        doi,
        title,
        abstract
    FROM citations
    WHERE run_id = (
        SELECT run_id
        FROM ingestion_runs
        ORDER BY retrieved_at DESC
        LIMIT 1
    )
    ORDER BY pmid
    """
).fetchall()

with output_path.open("w", encoding="utf-8", newline="") as file:
    writer = csv.DictWriter(
        file,
        fieldnames=[
            "run_id",
            "pmid",
            "doi",
            "title",
            "abstract",
            "decision",
            "reason_code",
            "justification",
            "reviewer",
        ],
    )

    writer.writeheader()

    for record in records:
        writer.writerow({
            **dict(record),
            "decision": "",
            "reason_code": "",
            "justification": "",
            "reviewer": "",
        })

connection.close()

print("Planilha criada em:", output_path)

Planilha criada em: /home/lucas/github/bibliometric-rag-assistant/data/curated/pubmed_sample_screening.csv
